In [ ]:
import pandas as pd
import os
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


#

# Model - 1 : Chart Summary
Qwen2.5-VL-7B-Instruct

In [ ]:
VL_MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"


qwen_processor = AutoProcessor.from_pretrained(VL_MODEL_NAME)


if 'qwen_model' not in globals() or qwen_model is None:
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        VL_MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    qwen_model.eval()
    print("Qwen Model loaded.")
else:
    print("Qwen Model already loaded, skipping.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Qwen Model loaded.


# Model - 2: Translation Model NLLB200

facebook/nllb-200-distilled-600M

In [ ]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

TRANS_MODEL_NAME = "facebook/nllb-200-distilled-600M"

trans_tokenizer = AutoTokenizer.from_pretrained(TRANS_MODEL_NAME)
if 'trans_model' not in globals() or trans_model is None:
  trans_model = AutoModelForSeq2SeqLM.from_pretrained(TRANS_MODEL_NAME)
  trans_model.eval()
  print("NLLB translation model loaded.")
else:
  print("NLLB translation model already loaded, skipping.")

def translate_en_to_bn(text):
    inputs = trans_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

    outputs = trans_model.generate(
        **inputs,
        forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids("ben_Beng"),
        max_new_tokens=256
    )

    return trans_tokenizer.decode(outputs[0], skip_special_tokens=True)


def summarize_and_translate(summary_en):
    return translate_en_to_bn(summary_en)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

NLLB translation model loaded.


## Pipeline: (Planner+insight_extractor+summarizer) = Qwen2.5-VL-7B-Instruct

Part - 2: Using Single Prompt For *Panner, Extactor, Summarizer* **AND** Single Call

In [ ]:
def generate_response_optimized(image, prompt_text, max_new_tokens=300):
    """Encode image ONCE, reuse for generation."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    text = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = qwen_processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(device)

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy = faster than sampling
            use_cache=True,           # KV cache reuse
            temperature=None,         # disable when do_sample=False
            top_p=None,               # disable when do_sample=False
        )

    new_tokens = output[0][input_len:]
    return qwen_processor.decode(new_tokens, skip_special_tokens=True).strip()


# ── Chain-of-Thought Single-Pass Prompt ──────────────────────────────────────
SINGLE_PASS_PROMPT = """You are an expert data analyst and data journalist.

Analyze the chart image carefully.

Think silently through these steps (DO NOT output them):
1. Identify chart type, domain, axes, units, and time range (if present)
2. Extract key data insights (trends, comparisons, patterns, anomalies)
3. Interpret domain meaning (causes, implications, real-world impact)

Then produce ONLY the final answer:

Summary:
Write a single, well-structured paragraph (4–6 sentences) that:
- Starts with the main trend or takeaway
- Includes at least one comparison or pattern
- Mentions any anomaly or notable feature (if present)
- Explains the real-world significance

Use clear, confident, natural English. No bullet points. No extra text."""


def run_pipeline_fast(image):
    print("Running single-pass optimized pipeline...")
    summary = generate_response_optimized(image, SINGLE_PASS_PROMPT, max_new_tokens=250)
    print("\n── Chart Summary ──")
    print(summary)
    return {"summary": summary}

# Load Old Dataset

In [ ]:
folder_path = "/content/drive/MyDrive/TWP/Image_Dataset/TestSet/UthfolToDO-2"

In [ ]:
image_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
dataset = [Image.open(f).convert("RGB") for f in image_files]

print(f"Loaded {len(dataset)} images into the dataset.")



Loaded 25 images into the dataset.


In [ ]:
df_images = pd.DataFrame({"image": image_files})
df_images['image_name'] = df_images['image'].apply(lambda x: os.path.basename(x))
# Drop the 'image' column (full path) as it's not needed, keeping only 'image_name'
df_images = df_images.drop(columns=['image'])
print(f"Size: {len(df_images)}")


Size: 25


In [ ]:
df_images.head()

,image_name
0,Test (201).png
1,Test (205).png
2,Test (204).png
3,Test (206).png
4,Test (202).png


## New Dataset Creation by Single Prompt technique

In [ ]:

summary_data = []

for i, image in enumerate(dataset):
    print(f"\nProcessing image {i+1}/{len(dataset)}...")

    # Generate English summary
    english_summary_result = run_pipeline_fast(image)
    english_summary = english_summary_result["summary"]

    # Generate Bangla summary
    bangla_summary = summarize_and_translate(english_summary)

    summary_data.append({
        "english_summary": english_summary,
        "bangla_summary": bangla_summary
    })

# Create a Pandas DataFrame
df_by_single_prompt = pd.DataFrame(summary_data)
df_by_single_prompt.head()



Processing image 1/25...
Running single-pass optimized pipeline...

── Chart Summary ──
The chart illustrates the significant increase in agricultural water withdrawal in Greece over the past five decades. Starting from around 2.5 billion cubic meters in 1970, withdrawals nearly tripled by 1990 to approximately 7.5 billion cubic meters before peaking at about 8.5 billion cubic meters in 2005. Thereafter, withdrawals declined slightly but remained above 8 billion cubic meters until 2015, when they began to stabilize. This trend highlights the growing pressure on water resources for agriculture, particularly during the mid-1990s and early 2000s, which could be attributed to increased irrigation practices and population growth. The decline after 2005 suggests potential improvements in water management or changes in agricultural practices, though further investigation is needed to confirm this. The high levels of water withdrawal underscore the critical need for sustainable water manageme

# Create Final Dataset & Save

In [ ]:
final_dataset = pd.concat([df_images, df_by_single_prompt], axis=1)
print("Final dataset created:")
final_dataset.head()


Final dataset created:


,image_name,english_summary,bangla_summary
0,Test (179).png,The chart illustrates the steady increase in G...,এই চার্টটি ২০০০ থেকে ২০১৮ সাল পর্যন্ত গানায় য...
1,Test (176).png,The chart illustrates the general government f...,চার্টটি ১৯৭০ থেকে ২০২০ সাল পর্যন্ত মরিশাসের জন...
2,Test (180).png,The chart illustrates the gross fixed capital ...,চার্টটি ১৯৭০ থেকে ২০২০ সাল পর্যন্ত মরিশাসের জন...
3,Test (181).png,The chart illustrates the gross graduation rat...,চার্টটি মারিশিয়সের উচ্চশিক্ষার জন্য 1925 থেকে...
4,Test (178).png,The chart illustrates the gross enrolment rati...,চার্টটি ১৯৭০ থেকে ২০২০ সাল পর্যন্ত মরিশাসের মা...


In [ ]:
output_folder_path = "/content/drive/MyDrive/TWP/Summary_Dataset/TestSet"

# Ensure the directory exists
os.makedirs(output_folder_path, exist_ok=True)

# Count existing .xlsx files in the directory
existing_files = [f for f in os.listdir(output_folder_path) if f.endswith('.xlsx')]
next_file_number = len(existing_files) + 1

# Create the new unique filename
output_file_name = f"testset_summaries_{next_file_number}.xlsx"
output_file_path = os.path.join(output_folder_path, output_file_name)

final_dataset.to_excel(output_file_path, index=False)
print(f"\nFinal dataset saved to {output_file_path}")


Final dataset saved to /content/drive/MyDrive/TWP/Summary_Dataset/TestSet/testset_summaries_7.xlsx
